# Trees and Ensembles, Notebook 2
## Decision trees: from the 14 firms to financial ratios

**Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

### What this is about

A decision tree is a chain of questions that ends in an answer:
"is debt low? yes: pays. No? does it have collateral?...". It is the most explainable model
there is: every prediction reads as a sentence, and a manager can argue about each question.

What the algorithm does on its own is **choose the questions**, and for that it uses what we saw in Notebook 1:
at each node it asks about the attribute with the highest information gain (ID3, Quinlan 1986) or
the largest reduction in Gini impurity (CART, Breiman and others, 1984). Today we build both: the
ID3 tree by hand, on the 14 firms, and the CART tree with `scikit-learn`, on 219 firms with
real financial ratios.

### What you will learn today

1. How ID3 builds a tree step by step using information gain.
2. What CART changes: the Gini index, binary splits on numeric variables and the
   search for the best threshold.
3. How to train, draw and read a tree with `scikit-learn`, and how to measure it on data it has not seen.
4. How the same mechanism predicts a number (regression tree) using variance.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
print("Ready.")


## 1. ID3 by hand: the tree of the 14 firms

The ID3 recipe fits in three lines: (1) if all the firms in the group have the same
answer, it is a leaf; (2) otherwise, compute the gain of each remaining attribute and ask
about the best one; (3) split the group by the answer and repeat on each branch. It is a
**greedy** algorithm: it picks the best question now, without checking whether another combination would give a
smaller tree.


In [ ]:

# The 14 firms that applied for credit (the small table from the course)
credito = pd.DataFrame({
    "tamano":    ["pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande"],
    "deuda":     ["baja", "baja", "baja", "baja", "alta", "alta", "alta", "alta", "alta", "media", "media", "media", "media", "media"],
    "garantia":  ["no", "si", "no", "si", "si", "si", "no", "no", "no", "no", "si", "si", "no", "no"],
    "historial": ["bueno", "malo", "malo", "bueno", "bueno", "malo", "bueno", "malo", "bueno", "bueno", "bueno", "malo", "malo", "malo"],
    "paga":      ["si", "si", "si", "si", "si", "si", "no", "no", "no", "si", "si", "no", "no", "no"],
})
credito.index = range(1, 15)
credito


In [ ]:

def entropia(serie):
    p = serie.value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())

def ganancia(df, objetivo, atributo):
    h_cond = sum(len(g) / len(df) * entropia(g[objetivo]) for _, g in df.groupby(atributo))
    return entropia(df[objetivo]) - h_cond

def id3(df, objetivo, atributos, nivel=0):
    # Builds the tree recursively and prints each decision
    sangria = "    " * nivel
    if df[objetivo].nunique() == 1:                          # pure group: leaf
        print(f"{sangria}-> leaf: {objetivo} = {df[objetivo].iloc[0]}  ({len(df)} firms)")
        return df[objetivo].iloc[0]
    if not atributos:                                        # no questions left: majority
        print(f"{sangria}-> leaf (majority): {df[objetivo].mode()[0]}  ({len(df)} firms)")
        return df[objetivo].mode()[0]
    ganancias = {a: ganancia(df, objetivo, a) for a in atributos}
    mejor = max(ganancias, key=ganancias.get)
    print(f"{sangria}[{len(df)} firms] gains: " + ", ".join(f"{a} {g:.3f}" for a, g in ganancias.items()) + f"  ->  question: {mejor}")
    ramas = {}
    for valor, grupo in df.groupby(mejor):
        print(f"{sangria}  {mejor} = {valor}:")
        ramas[valor] = id3(grupo, objetivo, [a for a in atributos if a != mejor], nivel + 1)
    return (mejor, ramas)

arbol_id3 = id3(credito, "paga", ["tamano", "deuda", "garantia", "historial"])


**How to read it.** At the root, debt wins (0.292 bits). The low-debt branch is pure
(everyone pays): a leaf. In the high-debt branch collateral wins; in the medium-debt branch,
credit history. Three questions, five leaves, and size is never asked: the tree discarded it
on its own. Notice that a different attribute wins on each branch: the tree asks different things depending on
the path.

With the tree, a new firm is predicted by following the questions:


In [ ]:

def predecir(arbol, empresa):
    while isinstance(arbol, tuple):
        atributo, ramas = arbol
        arbol = ramas[empresa[atributo]]
    return arbol

nueva = {"tamano": "grande", "deuda": "media", "garantia": "no", "historial": "malo"}
print("New firm:", nueva, "->  paga:", predecir(arbol_id3, nueva))
print("Accuracy on the 14 training firms:", np.mean([predecir(arbol_id3, e) == e["paga"] for _, e in credito.iterrows()]))


The tree gets all 14 firms it was built from right. That says nothing about new
firms: with 14 rows and 4 attributes, almost any table can be "explained" completely. Notebook 3
is devoted to that problem.

## 2. What CART changed: Gini, thresholds and binary splits

CART (Breiman, Friedman, Olshen and Stone, 1984) is the tree used by almost all current
software, `scikit-learn` included. It differs from ID3 in three things:

1. It measures impurity with the **Gini index** instead of entropy:
   $$\text{Gini} = 1 - p^2 - (1 - p)^2 = 2p(1 - p)$$
   where $p$ is the fraction of one class in the group. It is the probability that two firms
   drawn at random from the group belong to different classes: 0 in a pure group, 0.5 in a half-and-half
   one. It looks a lot like entropy and avoids the logarithm.
2. It always splits in **two**: for a numeric variable it looks for the best **threshold** ("sales/debt
   ≤ 1.28?") by trying every possible cut; for a categorical one, the best partition into two
   groups. Binary splits allow the same variable to be reused further down with another threshold.
3. It also works for **predicting numbers** (regression), using variance as the impurity.


In [ ]:

p = np.linspace(0, 1, 200)
gini = 2 * p * (1 - p)
with np.errstate(divide="ignore", invalid="ignore"):
    H = np.nan_to_num(-p * np.log2(p) - (1 - p) * np.log2(1 - p))
fig = go.Figure()
fig.add_trace(go.Scatter(x=p, y=gini, name="Gini"))
fig.add_trace(go.Scatter(x=p, y=H / 2, name="entropy / 2", line=dict(dash="dash")))
fig.update_layout(title="Gini and entropy (rescaled) as a function of the fraction p of one class", xaxis_title="p", yaxis_title="impurity")
fig.show()


## 3. The real data: 219 firms and five ratios


| Ratio | What it measures |
|---|---|
| `deuda_activos` | total debt / total assets (leverage) |
| `razon_corriente` | current assets / current liabilities (liquidity) |
| `ventas_deuda` | sales / total debt (ability to service debt) |
| `ln_activos` | logarithm of total assets (size) |
| `roa` | net income / total assets (profitability) |
| `impago` | **what we want to predict:** 1 if the firm defaulted, 0 otherwise |


And the usual ritual: split before looking. 146 firms for training and 73 for testing,
with the same partition used by the rest of the set.


In [ ]:

# The 219 firms with financial ratios (the default table from the course, with 5 of its ratios).
# It is pasted right here so the notebook does not depend on any external file.
from io import StringIO
IMPAGO_CSV = """deuda_activos,razon_corriente,ventas_deuda,ln_activos,roa,impago
0.374,2.337,1.436,14.697,0.073,0.0
0.435,2.638,1.385,14.877,0.042,0.0
0.443,2.099,0.452,14.931,0.016,0.0
0.23,2.506,4.217,15.586,-0.041,0.0
0.317,2.273,3.116,15.708,0.021,0.0
0.312,3.282,4.842,15.785,0.047,0.0
0.628,1.386,2.91,14.066,0.064,0.0
0.64,1.777,2.895,14.236,0.068,0.0
0.719,1.44,1.505,14.697,0.075,0.0
0.551,1.078,1.944,15.564,0.113,0.0
0.541,0.991,2.091,15.581,0.022,0.0
0.575,0.995,1.787,15.738,0.035,0.0
0.454,3.266,7.321,14.877,0.064,0.0
0.574,3.017,3.159,14.457,0.07,0.0
0.526,3.333,1.962,14.399,0.077,0.0
0.724,0.8,1.625,14.036,0.016,0.0
0.508,1.406,4.356,14.349,0.406,0.0
0.494,1.051,2.299,14.515,0.117,0.0
0.572,1.557,2.106,14.016,0.126,0.0
0.578,1.498,2.033,14.293,0.086,0.0
0.558,1.605,2.035,14.52,0.099,0.0
0.314,2.127,10.633,13.738,0.087,0.0
0.467,1.661,5.599,14.219,0.094,0.0
0.499,1.482,3.109,14.424,0.067,0.0
0.259,3.734,5.692,14.172,0.234,0.0
0.202,4.804,7.312,14.417,0.195,0.0
0.268,3.663,6.08,14.897,0.228,0.0
0.409,1.671,4.523,14.253,0.12,0.0
0.307,2.171,6.817,14.38,0.172,0.0
0.258,3.481,10.557,14.454,0.274,0.0
0.393,1.55,3.81,14.68,0.024,0.0
0.558,1.102,2.597,14.108,0.005,0.0
0.432,1.365,3.534,14.01,0.119,0.0
0.53,1.281,3.099,12.687,0.272,0.0
0.244,3.411,16.602,13.143,0.489,0.0
0.354,2.877,3.112,13.311,0.075,0.0
0.381,1.56,2.749,14.696,0.074,0.0
0.411,2.198,2.46,14.839,0.043,0.0
0.401,1.954,2.067,14.915,0.037,0.0
0.387,1.955,2.616,14.835,0.342,0.0
0.438,1.029,2.619,14.929,0.023,0.0
0.413,1.44,2.869,14.889,-0.011,0.0
0.891,1.118,2.023,13.415,0.036,0.0
1.052,0.741,1.8,14.355,-0.093,0.0
0.466,0.899,1.509,14.364,0.034,0.0
0.491,1.196,2.335,14.414,-0.017,0.0
0.505,1.053,2.678,14.631,0.076,0.0
0.354,2.45,3.08,14.453,0.084,0.0
0.464,1.843,2.521,14.559,0.119,0.0
0.566,1.286,1.675,14.838,0.111,0.0
0.543,1.766,2.619,14.933,0.029,0.0
0.855,1.131,1.805,15.005,0.029,0.0
0.558,1.744,2.516,15.074,0.028,0.0
0.736,1.153,1.564,14.292,0.054,0.0
0.708,1.192,1.352,14.432,0.034,0.0
0.657,1.269,1.395,14.387,0.027,0.0
0.653,1.061,1.588,14.303,0.04,0.0
0.619,1.005,2.113,14.235,0.038,0.0
0.197,9.632,5.15,14.586,0.032,0.0
0.284,1.461,6.585,13.479,0.11,0.0
0.3,1.438,5.251,13.541,0.012,0.0
0.283,2.928,5.778,13.561,0.016,0.0
0.476,1.788,5.329,13.143,0.123,0.0
0.47,1.759,5.033,13.329,0.13,0.0
0.459,2.104,4.49,13.372,0.076,0.0
0.388,2.409,5.319,13.283,0.201,0.0
0.547,1.979,2.09,13.806,0.15,0.0
0.568,1.812,1.937,13.914,0.071,0.0
0.461,1.923,1.499,14.059,-0.013,0.0
0.538,2.174,1.801,14.208,0.025,0.0
0.497,1.878,1.917,14.284,0.075,0.0
0.208,3.306,5.181,15.336,-0.003,0.0
0.241,3.005,4.709,15.428,0.011,0.0
0.222,3.11,3.776,15.427,0.021,0.0
0.392,2.268,6.865,12.39,0.334,0.0
0.252,5.229,7.751,12.318,0.176,0.0
0.849,1.56,1.326,12.304,-0.186,0.0
0.36,2.671,5.196,12.142,0.186,0.0
0.028,34.514,71.252,12.295,0.292,0.0
0.631,1.369,1.835,13.284,0.136,0.0
0.507,1.464,1.625,13.732,-0.217,0.0
0.578,1.577,2.553,13.963,0.02,0.0
0.448,1.681,0.566,13.768,0.024,0.0
0.363,2.721,2.762,13.301,0.446,0.0
0.403,2.199,2.673,13.506,0.067,0.0
0.4,2.237,1.139,13.576,0.043,0.0
0.432,1.579,2.39,14.7,0.066,0.0
0.464,1.763,2.067,14.843,0.059,0.0
0.475,1.707,1.833,14.95,0.06,0.0
0.089,8.849,16.344,15.39,0.013,0.0
0.125,7.175,11.171,15.513,0.059,0.0
0.125,7.156,12.795,15.55,0.041,0.0
0.539,1.764,3.733,15.384,0.088,0.0
0.502,1.79,3.797,15.397,0.035,0.0
0.522,1.572,2.624,15.391,0.029,0.0
0.77,0.383,1.984,15.834,-0.046,0.0
0.746,0.465,3.31,16.041,0.057,0.0
0.666,0.539,3.919,16.051,0.018,0.0
0.14,8.346,5.975,15.361,0.008,0.0
0.183,5.927,4.699,15.443,0.026,0.0
0.24,4.46,3.828,15.549,0.016,0.0
0.749,1.39,2.739,15.184,0.015,0.0
0.72,1.212,2.976,15.14,0.006,0.0
0.824,0.937,3.12,15.016,-0.089,0.0
0.127,0.463,3.018,13.706,0.036,0.0
0.584,3.254,1.629,14.581,0.053,0.0
0.565,0.66,1.621,14.753,0.085,0.0
0.853,3.41,0.45,13.939,0.012,0.0
0.836,4.713,0.337,14.072,0.007,0.0
0.968,2.708,0.249,14.346,-0.016,0.0
0.387,2.077,1.735,15.354,-0.039,0.0
0.398,2.395,2.933,15.532,0.059,0.0
0.327,2.428,1.813,15.518,0.036,0.0
0.635,1.999,0.061,15.292,-0.005,0.0
0.279,4.784,0.637,14.664,0.016,0.0
0.162,2.572,1.5,14.58,0.057,0.0
0.21,1.923,15.689,12.954,0.112,0.0
0.434,0.844,4.415,12.837,-0.246,0.0
0.271,1.323,5.378,12.849,0.17,0.0
0.416,2.122,5.392,13.508,0.098,0.0
0.409,2.225,4.438,13.477,0.018,0.0
0.262,3.185,5.914,13.392,0.128,0.0
0.23,4.172,5.796,14.956,0.218,0.0
0.085,11.333,15.115,15.14,0.246,0.0
0.134,6.66,9.398,15.292,0.101,0.0
0.759,2.52,2.074,13.84,0.084,0.0
0.77,1.92,1.572,14.162,0.031,0.0
0.675,2.107,1.621,13.957,0.036,0.0
0.601,2.387,1.541,13.442,-0.214,0.0
0.733,1.376,1.151,13.28,-0.11,0.0
0.822,1.461,1.286,13.478,-0.026,0.0
0.15,9.243,7.907,13.92,0.068,0.0
0.103,13.87,11.417,14.067,0.135,0.0
0.054,35.477,17.746,14.134,0.107,0.0
0.672,1.436,1.373,13.419,0.145,1.0
0.292,0.205,1.514,13.547,0.171,1.0
0.614,1.341,1.041,13.703,0.069,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,0.148,14.649,0.092,1.0
0.616,2.062,1.508,14.571,0.074,1.0
0.622,1.136,0.098,14.945,0.043,1.0
0.731,1.134,0.852,15.429,0.041,1.0
0.747,0.97,0.859,15.636,0.037,1.0
0.485,1.984,3.443,13.503,0.059,1.0
0.599,1.421,2.111,13.788,0.033,1.0
0.685,1.229,1.801,13.965,0.009,1.0
0.675,0.869,1.083,13.874,0.096,1.0
0.828,0.878,0.992,14.654,0.018,1.0
0.696,1.079,3.177,14.413,0.072,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.743,0.921,1.933,13.546,0.185,1.0
0.752,1.488,2.063,13.671,0.202,1.0
1.074,1.367,1.418,13.605,0.13,1.0
0.297,0.154,0.94,16.088,0.064,1.0
0.268,1.354,0.861,16.111,0.025,1.0
0.09,1.484,2.472,16.149,0.019,1.0
0.405,3.01,2.249,15.232,0.022,1.0
0.488,2.13,2.073,15.416,0.03,1.0
0.528,1.87,2.078,15.514,0.071,1.0
0.715,1.106,1.3,15.805,0.055,1.0
0.66,1.287,1.822,15.719,0.018,1.0
0.678,1.252,1.754,15.89,0.029,1.0
0.631,1.578,1.041,14.786,0.02,1.0
0.635,1.733,1.055,14.82,0.026,1.0
0.659,1.585,1.171,14.771,0.028,1.0
0.689,1.368,0.95,12.36,-0.033,1.0
0.464,1.238,1.069,12.662,-0.162,1.0
0.845,1.356,0.552,12.717,-0.352,1.0
0.548,1.741,0.697,16.077,0.002,1.0
0.595,1.234,0.758,16.058,0.0,1.0
0.768,0.919,0.355,16.536,-0.08,1.0
0.535,3.214,3.409,12.725,0.186,1.0
0.588,2.784,4.476,12.623,0.303,1.0
0.683,2.317,4.213,12.453,0.327,1.0
0.42,1.553,1.755,15.717,0.012,1.0
0.397,2.599,2.095,15.712,0.028,1.0
0.348,2.223,1.268,15.703,0.021,1.0
0.592,1.518,2.17,14.555,0.047,1.0
0.608,1.401,1.509,14.678,0.014,1.0
0.661,0.001,0.941,14.767,-0.013,1.0
1.48,0.225,0.001,12.488,-0.078,1.0
1.575,0.324,0.044,12.488,-0.009,1.0
0.638,0.87,1.662,15.163,-0.005,1.0
0.624,0.906,1.764,15.229,0.03,1.0
0.631,1.037,1.033,8.626,0.086,1.0
0.27,2.783,10.977,13.105,0.19,1.0
0.471,2.03,6.381,13.449,0.224,1.0
0.292,3.383,9.848,13.567,0.22,1.0
0.701,1.253,3.098,15.845,0.056,1.0
0.799,0.047,2.534,15.942,0.048,1.0
0.772,1.24,2.471,16.078,0.054,1.0
1.181,0.804,0.944,12.801,0.075,1.0
0.464,1.293,2.927,12.743,0.092,1.0
0.514,1.105,2.764,12.839,0.082,1.0
0.386,2.301,6.078,12.668,0.399,1.0
0.723,1.327,1.854,13.612,0.125,1.0
0.695,1.38,1.813,13.589,0.085,1.0
0.638,1.322,4.07,13.505,0.286,1.0
0.555,1.614,3.204,13.949,0.13,1.0
0.594,1.528,2.718,14.176,0.125,1.0
1.401,0.709,0.208,12.076,-0.146,1.0
0.916,0.616,1.27,12.629,0.316,1.0
0.75,0.489,2.46,12.725,0.29,1.0
0.445,9.164,0.0,11.167,-0.737,1.0
0.767,0.664,1.149,12.778,-0.779,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.775,1.025,3.294,13.079,0.112,1.0
0.731,0.932,4.179,12.889,0.126,1.0
0.842,0.673,1.911,13.274,-0.04,1.0
0.733,1.226,2.588,13.42,0.091,1.0
0.707,1.325,2.385,13.461,0.06,1.0
1.209,0.656,1.554,13.272,-0.565,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,1.479,14.649,0.092,1.0
0.616,2.062,2.277,14.571,0.074,1.0"""
impago = pd.read_csv(StringIO(IMPAGO_CSV))
print("Firms:", len(impago), "  Fraction in default:", round(impago["impago"].mean(), 3))
impago.describe().round(3).T[["mean", "min", "50%", "max"]]


In [ ]:

# The same partition in every notebook of the set: 146 firms for training, 73 for testing
RATIOS = ["deuda_activos", "razon_corriente", "ventas_deuda", "ln_activos", "roa"]
rng = np.random.default_rng(2)
orden = rng.permutation(len(impago))
es_prueba = np.zeros(len(impago), dtype=bool); es_prueba[orden[:73]] = True
impago["conjunto"] = np.where(es_prueba, "prueba", "entrenamiento")

X = impago[RATIOS]; y = impago["impago"]
X_train, y_train = X[~es_prueba], y[~es_prueba]
X_test, y_test = X[es_prueba], y[es_prueba]
print("Training:", len(X_train), "  Test:", len(X_test))
print("Fraction in default: training", round(y_train.mean(), 3), " test", round(y_test.mean(), 3))


### Finding the best split of a ratio, by hand

Before letting `scikit-learn` do it, let's see what it does. For a ratio, the candidate
thresholds are the midpoints between consecutive training values. For each one
we split into "less than or equal" and "greater", measure the Gini on each side and average, weighted by the
size of each side:

$$
G(u) = \frac{n_{izq} \cdot \text{Gini}_{izq} + n_{der} \cdot \text{Gini}_{der}}{n}
$$

(here $izq$ is the left side, $x \le u$, and $der$ the right side, $x > u$). The threshold with the lowest $G(u)$ is the best split of the ratio; the best ratio is the root.


In [ ]:

def gini_de(y):
    p = y.mean()
    return 2 * p * (1 - p)

def mejor_corte(x, y):
    # Tries every midpoint and returns (threshold, weighted Gini) of the best one
    valores = np.sort(x.unique())
    candidatos = (valores[:-1] + valores[1:]) / 2
    resultados = []
    for u in candidatos:
        izq = y[x <= u]; der = y[x > u]
        g = (len(izq) * gini_de(izq) + len(der) * gini_de(der)) / len(y)
        resultados.append((u, g))
    resultados = pd.DataFrame(resultados, columns=["umbral", "gini_ponderado"])
    return resultados.loc[resultados.gini_ponderado.idxmin()], resultados

print(f"Gini at the root (146 training firms, {y_train.mean():.1%} in default): {gini_de(y_train):.3f}")
print()
mejores = {}
for r in RATIOS:
    mejor, _ = mejor_corte(X_train[r], y_train)
    mejores[r] = mejor
    print(f"{r:16s} best threshold {mejor.umbral:8.3f}   weighted Gini {mejor.gini_ponderado:.3f}")
raiz = min(mejores, key=lambda r: mejores[r].gini_ponderado)
print()
print("Root:", raiz, "≤", round(mejores[raiz].umbral, 3))


In [ ]:

_, curva = mejor_corte(X_train["ventas_deuda"], y_train)
fig = px.line(curva, x="umbral", y="gini_ponderado", title="Weighted Gini as a function of where sales/debt is split (the minimum is the best split)")
fig.update_xaxes(range=[0, 8])
fig.show()


Sales/debt wins with threshold 1.278. Compare with ID3: there the root was chosen by entropy over
categorical values; here by Gini over a numeric threshold, and the mechanism is the same: split
and measure how much order you gain.

## 4. The tree with scikit-learn

`DecisionTreeClassifier` does exactly what we did above at every node, for every variable. We
limit it to depth 2 so it fits on a sheet of paper, and draw it.


In [ ]:

arbol2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)

plt.figure(figsize=(12, 5))
plot_tree(arbol2, feature_names=RATIOS, class_names=["pays", "defaults"], filled=True, rounded=True, fontsize=10, impurity=True, proportion=False)
plt.title("CART tree of depth 2 (146 training firms)")
plt.show()

print(export_text(arbol2, feature_names=RATIOS))


**How to read each box.** `gini` is the impurity of the group; `samples`, how many training
firms reached it; `value = [pays, defaults]`, how many of each class; `class`, the
majority, which is what the leaf predicts. The root is the one we found by hand. On the left
(low sales relative to debt) the next split is liquidity; on the right,
leverage.

**The only grade that counts is the one on the 73 test firms.**


In [ ]:

def evaluar(modelo, nombre):
    acc_tr = accuracy_score(y_train, modelo.predict(X_train))
    acc_te = accuracy_score(y_test, modelo.predict(X_test))
    print(f"{nombre:38s} training accuracy {acc_tr:.3f}   test accuracy {acc_te:.3f}")
    return acc_tr, acc_te

evaluar(arbol2, "tree of depth 2")
print()
cm = confusion_matrix(y_test, arbol2.predict(X_test))
print(pd.DataFrame(cm, index=["actual pays", "actual default"], columns=["predicted pays", "predicted default"]))
print()
print(f"Always predicting 'pays' would score {1 - y_test.mean():.3f} on test data: that is the floor to compare against.")


**What this is good for in business.** A CART tree on ratios is a credit *scoring* model
that reads like a rulebook: "if it sells less than 1.3 times its debt and its liquidity is below
1.9, high risk". The algorithm picked the threshold, but the manager can argue about it,
move it, or demand that a ratio not be used. That transparency is its value; its weakness is that
it is unstable (change two rows and the root may change) and that without a brake it memorizes the
data. Both things are the subject of the notebooks that follow.

### Entropy or Gini: does it change anything?

`scikit-learn` lets you choose the criterion. They almost always give the same tree or a very similar one.


In [ ]:

arbol2_entropia = DecisionTreeClassifier(max_depth=2, criterion="entropy", random_state=0).fit(X_train, y_train)
evaluar(arbol2, "depth 2, Gini criterion")
evaluar(arbol2_entropia, "depth 2, entropy criterion")
print()
print(export_text(arbol2_entropia, feature_names=RATIOS))


## 5. The same tree for predicting a number: regression

To predict profitability (`roa`) instead of "pays or not", the mechanism is identical, with a different
yardstick for impurity: the **variance** of the group (or, equivalently, its sum of squares
around the mean). The split that most reduces the weighted variance wins, and each leaf predicts
the mean of its group. The result is a step function: two means with one split,
four with two.

$$
V(u) = \frac{n_{izq} \cdot \text{Var}_{izq} + n_{der} \cdot \text{Var}_{der}}{n}
$$

We do it with a single variable, sales/debt, so we can draw the prediction.


In [ ]:

x_r = X_train["ventas_deuda"]; y_r = impago.loc[X_train.index, "roa"]
valores = np.sort(x_r.unique()); candidatos = (valores[:-1] + valores[1:]) / 2
filas = []
for u in candidatos:
    izq = y_r[x_r <= u]; der = y_r[x_r > u]
    filas.append((u, len(izq), izq.mean(), izq.var(ddof=0), len(der), der.mean(), der.var(ddof=0),
                  (len(izq) * izq.var(ddof=0) + len(der) * der.var(ddof=0)) / len(y_r)))
cortes = pd.DataFrame(filas, columns=["umbral", "n_izq", "promedio_izq", "var_izq", "n_der", "promedio_der", "var_der", "varianza_ponderada"])
mejor = cortes.loc[cortes.varianza_ponderada.idxmin()]
print(f"Variance of roa without splitting: {y_r.var(ddof=0):.5f}")
print(f"Best split: sales/debt ≤ {mejor.umbral:.3f}   weighted variance {mejor.varianza_ponderada:.5f}   "
      f"(explains {1 - mejor.varianza_ponderada / y_r.var(ddof=0):.1%} of the variance)")
print(f"Prediction: roa = {mejor.promedio_izq:.4f} if sales/debt ≤ {mejor.umbral:.3f}, and {mejor.promedio_der:.4f} if greater")
cortes.round(4).head(8)


In [ ]:

reg1 = DecisionTreeRegressor(max_depth=1, random_state=0).fit(X_train[["ventas_deuda"]], y_r)
reg3 = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X_train[["ventas_deuda"]], y_r)
grilla = pd.DataFrame({"ventas_deuda": np.linspace(0, 8, 400)})
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_r, y=y_r, mode="markers", name="training", opacity=0.5))
fig.add_trace(go.Scatter(x=grilla.ventas_deuda, y=reg1.predict(grilla), name="tree with 1 split (2 steps)", line=dict(width=3)))
fig.add_trace(go.Scatter(x=grilla.ventas_deuda, y=reg3.predict(grilla), name="tree of depth 3 (up to 8 steps)", line=dict(width=2, dash="dash")))
fig.update_layout(title="Regression tree: the prediction is a staircase", xaxis_title="sales / debt", yaxis_title="roa", xaxis_range=[0, 8])
fig.show()
print("The scikit-learn split at depth 1:", round(float(reg1.tree_.threshold[0]), 3), "(the same one we found by hand)")


A regression tree is the building block used by **gradient boosting** (Notebook 4): adding up many
small staircases, each fitted to what the previous ones left unexplained.

## 6. What to take away

- ID3 chooses at each node the attribute with the highest information gain; CART, the split with the largest
  reduction in Gini (classification) or variance (regression). The logic is the same: split and
  measure how much order you gain.
- A tree reads like a rulebook; that is its great value and the reason it is still used in credit,
  where every rejection has to be explained.
- Numeric splits are found by trying every possible threshold; the tree assumes
  no shape for the relationship (remember the U from Notebook 1).
- The grade that counts is the test grade. With 14 rows, a tree gets everything right and has learned nothing.

## Exercises

**Exercise 1.** Build by hand (with the `mejor_corte` function) the split of the left
branch of the root: take only the training firms with `ventas_deuda <= 1.278` and
find the best ratio and threshold. Does it match what `scikit-learn` drew?

**Exercise 2.** Train trees of depth 1, 2, 3, 5 and unlimited. Write down the training and
test accuracy of each one. Where do the two curves separate? (That is the subject of
Notebook 3; here you only need to see it.)

**Exercise 3.** Repeat the regression tree from section 5 using the other four ratios
(all except `roa`, which is what is being predicted) instead of just one, with depth 2. Which ratio
does it choose at the root? How much of the variance of `roa` does it explain on test data (`score` on the test firms)?


## Solutions


In [ ]:

# SOLUTION 1 -- the left branch
izq = X_train["ventas_deuda"] <= 1.278
for r in RATIOS:
    mejor, _ = mejor_corte(X_train.loc[izq, r], y_train[izq])
    print(f"{r:16s} threshold {mejor.umbral:8.3f}   weighted Gini {mejor.gini_ponderado:.3f}")
print("razon_corriente wins with threshold 1.866: the same split scikit-learn drew to the left of the root.")


In [ ]:

# SOLUTION 2 -- depth
for d in [1, 2, 3, 5, None]:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    evaluar(m, f"depth {d}  ({m.get_n_leaves()} leaves)")
print("Training accuracy always goes up; test accuracy stops rising at depth 2 or 3 and then falls: overfitting.")


In [ ]:

# SOLUTION 3 -- regression with the other ratios
SIN_ROA = ["deuda_activos", "razon_corriente", "ventas_deuda", "ln_activos"]
y_roa_train = impago.loc[X_train.index, "roa"]; y_roa_test = impago.loc[X_test.index, "roa"]
reg4 = DecisionTreeRegressor(max_depth=2, random_state=0).fit(X_train[SIN_ROA], y_roa_train)
print(export_text(reg4, feature_names=SIN_ROA))
print(f"R² on training {reg4.score(X_train[SIN_ROA], y_roa_train):.3f}   R² on test {reg4.score(X_test[SIN_ROA], y_roa_test):.3f}")
print("The root picks the ratio that best separates mean profitability. With 73 test firms the R² is noisy: compare it with the 15% of the single split.")
